# Event–Market Gold EDA
EDA riêng cho point-in-time join, event-window coverage và abnormal return.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
gold = pd.read_parquet(ROOT / 'data_lake/gold/news_market_impact')
for c in ['event_date_vn','t0_date','anchor_date_t_minus_1','session_date']:
    gold[c] = pd.to_datetime(gold[c], errors='coerce')
print(f'Gold rows: {len(gold):,}')

In [ ]:
pairs = gold.drop_duplicates(['event_id','ticker']).copy()
bad_t0 = np.where(pairs.published_after_close, pairs.t0_date <= pairs.event_date_vn, pairs.t0_date < pairs.event_date_vn)
quality = pd.Series({
    'rows': len(gold),
    'event_ticker_pairs': len(pairs),
    'events': gold.event_id.nunique(),
    'tickers': gold.ticker.nunique(),
    'duplicate_window_keys': gold.duplicated(['event_id','ticker','relative_session']).sum(),
    'published_after_close': pairs.published_after_close.sum(),
    'bad_t0': bad_t0.sum(),
    'bad_anchor': (pairs.anchor_date_t_minus_1 >= pairs.t0_date).sum(),
}, name='value').to_frame()
display(quality)

In [ ]:
coverage = gold.groupby('relative_session').agg(
    requested=('event_id','size'),
    available=('has_market_data','sum'),
)
coverage['availability_pct'] = (100*coverage.available/coverage.requested).round(2)
display(coverage)
coverage.availability_pct.plot.bar(figsize=(9,4), ylim=(0,105), title='Market coverage theo relative session (%)', rot=0)
plt.tight_layout(); plt.show()

In [ ]:
available = gold[gold.has_market_data & gold.abnormal_return.notna()].copy()
stats = available.groupby('relative_session').abnormal_return.agg(['count','mean','median','std'])
stats['se'] = stats['std']/np.sqrt(stats['count'])
stats['ci95_low'] = stats['mean'] - 1.96*stats['se']
stats['ci95_high'] = stats['mean'] + 1.96*stats['se']
display((100*stats).assign(count=stats['count']).round(3))
fig, ax = plt.subplots(figsize=(9,5))
x = stats.index.to_numpy()
mean = (100*stats['mean']).to_numpy()
error = (100*1.96*stats['se']).to_numpy()
ax.errorbar(x, mean, yerr=error, marker='o', capsize=4)
ax.axhline(0,color='black',linewidth=.8)
ax.set(title='Mean cumulative abnormal return', xlabel='Relative session', ylabel='%')
plt.tight_layout(); plt.show()

In [ ]:
focus = available[available.relative_session.isin([0,1,3,5])]
timing = focus.groupby(['published_after_close','relative_session']).agg(
    events=('event_id','size'), mean_abnormal=('abnormal_return','mean'),
).reset_index()
timing['mean_abnormal_pct'] = (100*timing.mean_abnormal).round(3)
display(timing)
ticker_t5 = available[available.relative_session.eq(5)].groupby('ticker').agg(
    events=('event_id','size'), mean_abnormal=('abnormal_return','mean'),
).query('events >= 20').sort_values('mean_abnormal')
ticker_t5['mean_abnormal_pct'] = (100*ticker_t5.mean_abnormal).round(3)
display(ticker_t5[['events','mean_abnormal_pct']])
print('Cảnh báo: các thống kê này mô tả tương quan, không chứng minh tác động nhân quả.')